# CS3807 – Deep Learning Laboratory — Experiment 5
## Comprehensive Study of CNN Training, Regularization, Optimization, Hyperparameter Tuning, Transfer Learning and Cross-Validation

**Architecture:** MobileNetV2  |  **Dataset:** Oxford-IIIT Pet (37 breeds)

Run cells top-to-bottom in Google Colab. Set **Runtime → Change runtime type → GPU** before running.

> Epoch counts below are kept small (5–8) for lab-time feasibility. Increase `EPOCHS_*` variables for better curves if you have time/GPU quota.

## 1. Setup & Imports

In [ ]:
!pip install -q tensorflow_datasets

import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import time

from tensorflow.keras import layers, models, optimizers, regularizers
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

print("TF version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

tf.random.set_seed(42)
np.random.seed(42)

## 2. Dataset and Experimental Setup — Oxford-IIIT Pet

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 37

(ds_train_full, ds_test), ds_info = tfds.load(
    'oxford_iiit_pet',
    split=['train', 'test'],
    with_info=True,
    as_supervised=False
)

class_names = ds_info.features['label'].names

def preprocess(example):
    image = tf.image.resize(example['image'], (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)          # normalize per MobileNetV2 requirements
    label = example['label']
    return image, label

# Train / Validation split (85/15). Test set stays untouched until Section 12.
ds_train_full = ds_train_full.shuffle(1000, seed=42)
train_size = int(0.85 * ds_info.splits['train'].num_examples)
ds_train_raw = ds_train_full.take(train_size)
ds_val_raw = ds_train_full.skip(train_size)

def make_pipeline(ds, batch_size=BATCH_SIZE, shuffle=False):
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(1000, seed=42)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_pipeline(ds_train_raw, shuffle=True)
val_ds = make_pipeline(ds_val_raw)
test_ds = make_pipeline(ds_test)

print("Train batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Val batches:", tf.data.experimental.cardinality(val_ds).numpy())
print("Test batches:", tf.data.experimental.cardinality(test_ds).numpy())

In [ ]:
def plot_curves(histories_dict, metric='loss', title='', ylabel=None):
    plt.figure(figsize=(7, 5))
    for label, hist in histories_dict.items():
        plt.plot(hist.history[metric], label=label, marker='o', markersize=3)
    plt.title(title)
    plt.xlabel('Epoch')
    plt.ylabel(ylabel if ylabel else metric)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

## 3. Weight Initialization (Section 5)

A small custom CNN (trained from scratch) is used here rather than the pretrained MobileNetV2, because
weight-initialization effects (especially zero-initialization failure) are only visible when weights are
initialized and trained from scratch.

In [ ]:
def build_small_cnn(initializer, num_classes=NUM_CLASSES):
    model = models.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.Conv2D(32, 3, activation='relu', padding='same', kernel_initializer=initializer),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu', padding='same', kernel_initializer=initializer),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu', padding='same', kernel_initializer=initializer),
        layers.MaxPooling2D(),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu', kernel_initializer=initializer),
        layers.Dense(num_classes, activation='softmax', kernel_initializer=initializer)
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

initializers_dict = {
    'Zero': tf.keras.initializers.Zeros(),
    'Random': tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.05, seed=42),
    'Xavier/Glorot': tf.keras.initializers.GlorotUniform(seed=42),
    'He': tf.keras.initializers.HeNormal(seed=42),
}

EPOCHS_INIT = 8
init_histories = {}
for name, init in initializers_dict.items():
    print(f"\n--- Training with {name} initialization ---")
    model = build_small_cnn(init)
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_INIT, verbose=1)
    init_histories[name] = hist

In [ ]:
plot_curves(init_histories, metric='loss', title='Plot 1: Training Loss vs. Epoch (Initialization)', ylabel='Training Loss')
plot_curves(init_histories, metric='val_accuracy', title='Plot 2: Validation Accuracy vs. Epoch (Initialization)', ylabel='Validation Accuracy (%)')

**Inference (fill in after running):**
- What does the plot show?
- What trend is observed (e.g. does Zero-init stay flat)?
- Why does this trend occur?

## 4. Regularization and Overfitting (Section 6) + Batch Normalization (Section 7)

In [ ]:
def build_reg_cnn(reg_type='none'):
    l2_reg = regularizers.l2(1e-4) if reg_type == 'l2' else None
    use_dropout = reg_type == 'dropout'
    use_bn = reg_type == 'batchnorm'

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Conv2D(32, 3, padding='same', kernel_regularizer=l2_reg)(inputs)
    if use_bn: x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding='same', kernel_regularizer=l2_reg)(x)
    if use_bn: x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding='same', kernel_regularizer=l2_reg)(x)
    if use_bn: x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu', kernel_regularizer=l2_reg)(x)
    if use_dropout: x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

reg_configs = ['none', 'l2', 'dropout', 'batchnorm']
EPOCHS_REG = 8
reg_histories = {}
for cfg in reg_configs:
    print(f"\n--- Training with regularization: {cfg} ---")
    model = build_reg_cnn(cfg)
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_REG, verbose=1)
    reg_histories[cfg] = hist

In [ ]:
# Plot 3: Training and Validation Accuracy vs Epoch (one figure per configuration)
for cfg, hist in reg_histories.items():
    plt.figure(figsize=(6, 4))
    plt.plot(hist.history['accuracy'], label='Train Accuracy')
    plt.plot(hist.history['val_accuracy'], label='Val Accuracy')
    plt.title(f'Plot 3: Accuracy vs Epoch ({cfg})')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy (%)'); plt.legend(); plt.grid(alpha=0.3)
    plt.show()

In [ ]:
# Plot 4: Training and Validation Loss vs Epoch
for cfg, hist in reg_histories.items():
    plt.figure(figsize=(6, 4))
    plt.plot(hist.history['loss'], label='Train Loss')
    plt.plot(hist.history['val_loss'], label='Val Loss')
    plt.title(f'Plot 4: Loss vs Epoch ({cfg})')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(alpha=0.3)
    plt.show()

In [ ]:
# Plot 5: With vs Without Batch Normalization (validation accuracy)
bn_compare = {'Without BN': reg_histories['none'], 'With BN': reg_histories['batchnorm']}
plot_curves(bn_compare, metric='val_accuracy', title='Plot 5: With vs. Without Batch Normalization', ylabel='Validation Accuracy (%)')

### Batch Normalization — Numerical Example (Section 7)

In [ ]:
x = np.array([2, 4, 6, 8], dtype=float)
mu = x.mean()
var = x.var()
eps = 1e-8
std = np.sqrt(var + eps)
x_hat = (x - mu) / std
gamma, beta = 1.0, 0.0
y = gamma * x_hat + beta

print("Batch mean (mu_B):", mu)
print("Batch variance (sigma_B^2):", var)
print("Normalized activations (x_hat):", np.round(x_hat, 3))
print("Output y = gamma*x_hat + beta:", np.round(y, 3))

## 5. Optimization Algorithms (Section 8)

In [ ]:
def build_cnn_for_opt():
    return build_reg_cnn('batchnorm')  # BN-equipped CNN, no other regularization

optimizer_dict = {
    'SGD': optimizers.SGD(learning_rate=0.01),
    'Momentum': optimizers.SGD(learning_rate=0.01, momentum=0.9),
    'RMSProp': optimizers.RMSprop(learning_rate=0.001),
    'Adam': optimizers.Adam(learning_rate=0.001),
}

EPOCHS_OPT = 8
opt_histories = {}
opt_results = []
for name, opt in optimizer_dict.items():
    print(f"\n--- Training with optimizer: {name} ---")
    model = build_cnn_for_opt()
    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_OPT, verbose=1)
    elapsed = time.time() - start
    opt_histories[name] = hist
    best_val_acc = max(hist.history['val_accuracy'])
    epoch_conv = int(np.argmax(hist.history['val_accuracy'])) + 1
    opt_results.append({
        'Optimizer': name,
        'Final Loss': round(hist.history['loss'][-1], 4),
        'Best Val. Accuracy': round(best_val_acc, 4),
        'Epoch to Converge': epoch_conv,
        'Time (s)': round(elapsed, 2),
    })

In [ ]:
plot_curves(opt_histories, metric='loss', title='Plot 6: Training Loss vs. Epoch (Optimizers)', ylabel='Training Loss')
plot_curves(opt_histories, metric='val_accuracy', title='Plot 7: Validation Accuracy vs. Epoch (Optimizers)', ylabel='Validation Accuracy (%)')

opt_df = pd.DataFrame(opt_results)
opt_df  # Optimizer comparison table for the report

## 6. CNN Hyperparameter Tuning (Section 9)

In [ ]:
def train_with_hparams(lr=0.001, batch_size=32, dropout=0.5, optimizer_name='adam', epochs=5):
    tr_ds = make_pipeline(ds_train_raw, batch_size=batch_size, shuffle=True)
    va_ds = make_pipeline(ds_val_raw, batch_size=batch_size)

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    model = models.Model(inputs, outputs)

    opt = optimizers.Adam(learning_rate=lr) if optimizer_name == 'adam' else optimizers.SGD(learning_rate=lr)
    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    hist = model.fit(tr_ds, validation_data=va_ds, epochs=epochs, verbose=0)
    return max(hist.history['val_accuracy'])

# NOTE: one hyperparameter is changed at a time (Section 9 experimental rule); others stay at defaults.

In [ ]:
# Plot 8: Learning Rate vs Validation Accuracy
lrs = [0.001, 0.0001]
lr_results = [train_with_hparams(lr=lr, epochs=5) for lr in lrs]

plt.figure(figsize=(6, 4))
plt.plot(lrs, lr_results, marker='o')
plt.xscale('log')
plt.title('Plot 8: Learning Rate vs. Validation Accuracy')
plt.xlabel('Learning Rate'); plt.ylabel('Validation Accuracy (%)'); plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Plot 9: Batch Size vs Validation Accuracy
batch_sizes = [16, 32, 64]
bs_results = [train_with_hparams(batch_size=bs, epochs=5) for bs in batch_sizes]

plt.figure(figsize=(6, 4))
plt.plot(batch_sizes, bs_results, marker='o')
plt.title('Plot 9: Batch Size vs. Validation Accuracy')
plt.xlabel('Batch Size'); plt.ylabel('Validation Accuracy (%)'); plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Plot 10: Dropout Rate vs Validation Accuracy
dropouts = [0, 0.25, 0.5]
do_results = [train_with_hparams(dropout=d, epochs=5) for d in dropouts]

plt.figure(figsize=(6, 4))
plt.plot(dropouts, do_results, marker='o')
plt.title('Plot 10: Dropout Rate vs. Validation Accuracy')
plt.xlabel('Dropout Rate'); plt.ylabel('Validation Accuracy (%)'); plt.grid(alpha=0.3)
plt.show()

## 7. Transfer Learning and Fine-Tuning (Section 10) — MobileNetV2 pretrained on ImageNet

In [ ]:
def build_transfer_model(fine_tune=False, fine_tune_at=100, fine_tune_lr=1e-5):
    base_model = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
    base_model.trainable = fine_tune
    if fine_tune:
        for layer in base_model.layers[:fine_tune_at]:
            layer.trainable = False   # keep lower layers frozen; unfreeze upper layers only

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base_model(inputs, training=fine_tune)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    model = models.Model(inputs, outputs)

    lr = fine_tune_lr if fine_tune else 1e-3
    model.compile(optimizer=optimizers.Adam(learning_rate=lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

EPOCHS_TL = 6

print("--- Case A: Feature Extraction (frozen base) ---")
fe_model = build_transfer_model(fine_tune=False)
fe_hist = fe_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_TL, verbose=1)

print("\n--- Case B: Fine-Tuning (partial unfreeze + small LR) ---")
ft_model = build_transfer_model(fine_tune=True, fine_tune_at=100, fine_tune_lr=1e-5)
ft_hist = ft_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_TL, verbose=1)

In [ ]:
# Plot 11: Feature Extraction vs Fine-Tuning
tl_histories = {'Feature Extraction': fe_hist, 'Fine-Tuning': ft_hist}
plot_curves(tl_histories, metric='val_accuracy', title='Plot 11: Feature Extraction vs. Fine-Tuning', ylabel='Validation Accuracy (%)')

# Plot 12: Training and Validation Loss (before vs after fine-tuning)
plt.figure(figsize=(7, 5))
plt.plot(fe_hist.history['loss'], label='FE Train Loss')
plt.plot(fe_hist.history['val_loss'], label='FE Val Loss')
plt.plot(ft_hist.history['loss'], label='FT Train Loss')
plt.plot(ft_hist.history['val_loss'], label='FT Val Loss')
plt.title('Plot 12: Training and Validation Loss (Feature Extraction vs Fine-Tuning)')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

**Discussion:** Fine-tuning normally uses a much smaller learning rate than training a freshly-initialized
classifier because the pretrained convolutional filters already encode useful, well-tuned representations;
large gradient updates would destroy this pretrained knowledge ("catastrophic forgetting"), so small, careful
updates are used instead.

## 8. K-Fold Cross-Validation (Section 11)

Four candidate configurations (`C1`–`C4`) are selected from the studies above and evaluated with 5-fold CV
on the **training data only**. The independent test set remains untouched.

> Converting the full training split to NumPy arrays needs a few GB of RAM. If Colab runs out of memory, reduce
> `CV_SAMPLE_LIMIT` below to a smaller subset (e.g. 1500) for a lighter-weight demonstration.

In [ ]:
CV_SAMPLE_LIMIT = None   # set e.g. 1500 to subsample if you hit memory limits

def dataset_to_numpy(ds, limit=None):
    images, labels = [], []
    for img, lbl in tfds.as_numpy(ds):
        images.append(img)
        labels.append(lbl)
        if limit and len(images) >= limit:
            break
    return np.array(images), np.array(labels)

print("Collecting training data as NumPy arrays for K-Fold CV (this may take a while)...")
train_full_pipeline = ds_train_full.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
X_all, y_all = dataset_to_numpy(train_full_pipeline, limit=CV_SAMPLE_LIMIT)
print("Data shape:", X_all.shape, y_all.shape)

In [ ]:
configs = [
    {'name': 'C1', 'lr': 1e-3, 'dropout': 0.5,  'fine_tune': False},
    {'name': 'C2', 'lr': 1e-4, 'dropout': 0.5,  'fine_tune': False},
    {'name': 'C3', 'lr': 1e-3, 'dropout': 0.25, 'fine_tune': True},
    {'name': 'C4', 'lr': 1e-4, 'dropout': 0.25, 'fine_tune': True},
]

def build_cv_model(cfg):
    base_model = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
    base_model.trainable = cfg['fine_tune']
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base_model(inputs, training=cfg['fine_tune'])
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(cfg['dropout'])(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    model = models.Model(inputs, outputs)
    model.compile(optimizer=optimizers.Adam(learning_rate=cfg['lr']),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

K = 5
EPOCHS_CV = 4
kf = KFold(n_splits=K, shuffle=True, random_state=42)

cv_results = {}
for cfg in configs:
    fold_accs = []
    print(f"\n=== Cross-Validating {cfg['name']} ===")
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_all)):
        X_tr, X_val = X_all[train_idx], X_all[val_idx]
        y_tr, y_val = y_all[train_idx], y_all[val_idx]
        model = build_cv_model(cfg)
        model.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                  epochs=EPOCHS_CV, batch_size=32, verbose=0)
        _, val_acc = model.evaluate(X_val, y_val, verbose=0)
        fold_accs.append(val_acc)
        print(f"  Fold {fold_idx + 1}: Val Accuracy = {val_acc:.4f}")
    cv_results[cfg['name']] = fold_accs

In [ ]:
cv_summary = []
for name, accs in cv_results.items():
    row = {'Configuration': name}
    row.update({f'F{i+1}': round(a, 4) for i, a in enumerate(accs)})
    row['Mean ± SD'] = f"{np.mean(accs):.4f} ± {np.std(accs):.4f}"
    cv_summary.append(row)
cv_df = pd.DataFrame(cv_summary)
cv_df

In [ ]:
# Plot 13: 5-Fold Cross-Validation Accuracy with SD error bars
names = [c['name'] for c in configs]
means = [np.mean(cv_results[n]) for n in names]
sds = [np.std(cv_results[n]) for n in names]

plt.figure(figsize=(7, 5))
plt.bar(names, means, yerr=sds, capsize=6, color='skyblue')
plt.title('Plot 13: 5-Fold CV Accuracy by Configuration')
plt.xlabel('Hyperparameter Configuration'); plt.ylabel('Mean Validation Accuracy (%)')
plt.grid(alpha=0.3, axis='y')
plt.show()

best_config_name = max(cv_results, key=lambda n: np.mean(cv_results[n]))
best_config = next(c for c in configs if c['name'] == best_config_name)
print("Best configuration selected by CV:", best_config)

## 9. Final Model Evaluation (Section 12)

In [ ]:
print("Retraining the best configuration on the complete training data...")
final_model = build_cv_model(best_config)

start = time.time()
final_hist = final_model.fit(train_ds, validation_data=val_ds, epochs=8, verbose=1)
train_time = time.time() - start

test_loss, test_acc = final_model.evaluate(test_ds, verbose=1)
print(f"\nTest Accuracy: {test_acc:.4f}")

In [ ]:
y_true, y_pred, all_test_images = [], [], []
for imgs, labels in test_ds:
    preds = final_model.predict(imgs, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
    all_test_images.extend(imgs.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
num_params = final_model.count_params()

print("=== Final Model Report ===")
print(f"Mean CV Accuracy   : {np.mean(cv_results[best_config_name]):.4f}")
print(f"CV Standard Dev.    : {np.std(cv_results[best_config_name]):.4f}")
print(f"Test Accuracy       : {test_acc:.4f}")
print(f"Precision (weighted): {precision:.4f}")
print(f"Recall (weighted)   : {recall:.4f}")
print(f"F1-score (weighted) : {f1:.4f}")
print(f"Training Time       : {train_time:.2f}s")
print(f"Number of Parameters: {num_params:,}")

In [ ]:
# Plot 14: Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Plot 14: Confusion Matrix')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=90, fontsize=6); plt.yticks(fontsize=6)
plt.tight_layout()
plt.show()

# Identify most-confused class pairs
cm_off_diag = cm.copy()
np.fill_diagonal(cm_off_diag, 0)
top_confusions = np.dstack(np.unravel_index(np.argsort(-cm_off_diag.ravel())[:5], cm_off_diag.shape))[0]
print("Most frequently confused class pairs (true -> predicted):")
for t, p in top_confusions:
    if cm_off_diag[t, p] > 0:
        print(f"  {class_names[t]} -> {class_names[p]}: {cm_off_diag[t, p]} images")

In [ ]:
# Optional Plot 15: Misclassified Images
misclassified = [i for i in range(len(y_true)) if y_true[i] != y_pred[i]]
print(f"Total misclassified: {len(misclassified)} / {len(y_true)}")

sample_indices = misclassified[:6]
plt.figure(figsize=(15, 8))
for plot_i, idx in enumerate(sample_indices):
    img = all_test_images[idx]
    img_display = (img - img.min()) / (img.max() - img.min() + 1e-8)
    plt.subplot(2, 3, plot_i + 1)
    plt.imshow(img_display)
    plt.title(f"True: {class_names[y_true[idx]]}\nPred: {class_names[y_pred[idx]]}", fontsize=9)
    plt.axis('off')
plt.suptitle('Plot 15: Representative Misclassified Images')
plt.tight_layout()
plt.show()

## 10. Overall Results Summary (Section 13)

In [ ]:
overall_results = pd.DataFrame([
    {'Configuration': 'Baseline',              'CV Accuracy': None, 'SD': None, 'Test Accuracy': None, 'Training Time': None},
    {'Configuration': 'Best Initialization',     'CV Accuracy': None, 'SD': None, 'Test Accuracy': None, 'Training Time': None},
    {'Configuration': 'Best Regularization',     'CV Accuracy': None, 'SD': None, 'Test Accuracy': None, 'Training Time': None},
    {'Configuration': 'Best Optimizer',          'CV Accuracy': None, 'SD': None, 'Test Accuracy': None, 'Training Time': None},
    {'Configuration': 'Best Hyperparameters',    'CV Accuracy': None, 'SD': None, 'Test Accuracy': None, 'Training Time': None},
    {'Configuration': 'Fine-Tuned Model',
     'CV Accuracy': np.mean(cv_results[best_config_name]),
     'SD': np.std(cv_results[best_config_name]),
     'Test Accuracy': test_acc,
     'Training Time': train_time},
])
# Fill in the remaining rows manually from the results produced in Sections 3-7 above.
overall_results

## 11. Discussion Questions (Section 15) & Additional Exercise (Section 16)

Use the plots and tables generated above to answer the 23 discussion questions in the lab sheet, and to
complete the additional exercise (select two new hyperparameter/fine-tuning combinations, evaluate with
5-fold CV using the `build_cv_model` + `KFold` pattern from Section 8, and compare against `best_config`).